In [4]:

import pandas as pd
from pathlib import Path

# Arquivos de entrada e saída
arquivo_parquet = Path("PENSE2024_MICRODADOS_SELECIONADAS.parquet")
arquivo_dicionario = Path("Dicionário_Microdados_PENSE2024.xlsx")

arquivo_saida = Path("PENSE2024_MICRODADOS_COM_LABELS.parquet")

# Lê a base de dados
df = pd.read_parquet(arquivo_parquet)

# Lê o dicionário de variáveis
dic = pd.read_excel(
    arquivo_dicionario,
    sheet_name="PeNSE2024"
)

# Remove espaços dos códigos e descrições
dic["Nome da Variável"] = (
    dic["Nome da Variável"]
    .astype(str)
    .str.strip()
)

dic["Descrição da Variável"] = (
    dic["Descrição da Variável"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Cria o dicionário código: descrição
mapa_labels = {}

for _, linha in dic.iterrows():

    codigo = linha["Nome da Variável"]
    descricao = linha["Descrição da Variável"]

    if codigo and codigo.lower() != "nan" and descricao:
        mapa_labels[codigo] = descricao

# Renomeia as colunas da base
# Se não encontrar descrição, mantém o código original
novos_nomes = [
    mapa_labels.get(coluna, coluna)
    for coluna in df.columns
]

# Evita nomes duplicados
contagem = {}
nomes_finais = []

for nome in novos_nomes:

    contagem[nome] = contagem.get(nome, 0) + 1

    if contagem[nome] == 1:
        nomes_finais.append(nome)
    else:
        nomes_finais.append(f"{nome} ({contagem[nome]})")

# Aplica os novos nomes
df.columns = nomes_finais

# Grava novo arquivo Parquet
df.to_parquet(
    arquivo_saida,
    engine="pyarrow",
    compression="snappy",
    index=False
)

# Relatório
print("Renomeação concluída!")
print("-" * 50)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

print("\nNovos nomes das variáveis:")

for nome in df.columns:
    print(nome)

print(f"\nArquivo salvo: {arquivo_saida.resolve()}")


FileNotFoundError: [Errno 2] No such file or directory: 'PENSE2024_MICRODADOS_SELECIONADAS.parquet'